In [1]:
%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from transformers import AutoImageProcessor, AutoModel
from transformers.image_utils import load_image

from PIL import Image
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import clear_output

# Load DINOv3 Base with 16x16 patch projection
model_name = "facebook/dinov3-vitb16-pretrain-lvd1689m"
print(f"Loading DINOv3 checkpoint: {model_name}...")


model = AutoModel.from_pretrained(model_name)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
model = model.to(device)
model.eval()
print(f"DINOv3 loaded onto {device}!")

Loading DINOv3 checkpoint: facebook/dinov3-vitb16-pretrain-lvd1689m...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

DINOv3 loaded onto mps!


In [2]:
def extract_dinov3_features(pil_image, model, processor, device):
    # 3. Forward pass
    inputs = processor(images=pil_image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(**inputs)
        
    last_hidden_state = outputs.last_hidden_state # (1, Seq_Len, Hidden_Dim)
    
    # 4. Dynamically isolate spatial tokens
    # DINOv3 uses 1 CLS token + 4 register tokens.
    # At 224x224 with patch_size=16, we get a 14x14 = 196 patch grid.
    patch_size = 16
    target_h = processor.size["height"]
    target_w = processor.size["width"]
    grid_h = target_h // patch_size
    grid_w = target_w // patch_size
    num_spatial_patches = grid_h * grid_w
    
    # Slice the exact spatial patch tokens from the tail of the sequence
    patch_tokens = last_hidden_state[0, -num_spatial_patches:, :] 
    patch_grid = patch_tokens.reshape(grid_h, grid_w, -1)
    
    # L2 Normalize for robust cosine similarity calculation
    patch_grid_norm = F.normalize(patch_grid, p=2, dim=-1).cpu()
    
    # Reconstruct the precise image evaluated by the model (ensures flawless alignment)
    model_input_tensor = inputs["pixel_values"][0].cpu()
    mean = torch.tensor(processor.image_mean).view(3, 1, 1)
    std = torch.tensor(processor.image_std).view(3, 1, 1)
    unnormalized_tensor = (model_input_tensor * std) + mean
    display_image = unnormalized_tensor.permute(1, 2, 0).numpy()
    display_image = np.clip(display_image, 0, 1)
    
    return patch_grid_norm, display_image, grid_h, grid_w


def pil_from_array(image_array, model, processor, device):
    """
    Safely handles [1, H, W] grayscale arrays, passes them through DINOv3,
    and returns perfectly aligned spatial patch features.
    """
    # 1. Handle the [1, H, W] or [H, W] single-channel shape
    image_array = np.squeeze(image_array)
    if len(image_array.shape) != 2:
        raise ValueError(f"Expected a 2D grayscale slice, got shape {image_array.shape}")
        
    # 2. Convert grayscale values to 0-255 uint8 range
    if image_array.max() <= 1.0:
        image_array = (image_array * 255).astype(np.uint8)
    else:
        image_array = image_array.astype(np.uint8)
        
    # Expand 1-channel to 3-channels for the visual encoder
    rgb_image = np.stack([image_array] * 3, axis=-1)
    pil_image = Image.fromarray(rgb_image)
    return pil_image


def fake_pil_image():
    # 1. Create a mock scientific image with shape [1, H, W] containing distinct targets
    np.random.seed(42)
    h, w = 600, 800
    mock_sensor_data = np.zeros((1, h, w))
    y_idx, x_idx = np.indices((h, w))
    
    # Generate two distinct signal blobs
    mock_sensor_data[0] += 0.8 * np.exp(-((x_idx - 100)**2 + (y_idx - 150)**2) / (35**2))
    mock_sensor_data[0] += 0.6 * np.exp(-((x_idx - 260)**2 + (y_idx - 90)**2) / (25**2))
    mock_sensor_data[0] += np.random.normal(0, 0.04, (h, w)) # Sensor noise
    mock_sensor_data = np.clip(mock_sensor_data, 0, 1)
    return pil_from_array(mock_sensor_data)

In [23]:
def plot_cosine_similarity(x_patch, y_patch):
    # Clear previous image frame to unfreeze sliders
    clear_output(wait=True)
    
    # Select the query vector
    query_vector = patch_grid[y_patch, x_patch]
    
    # Dot product calculation for cosine similarity
    similarity_map = torch.einsum('hwd,d->hw', patch_grid, query_vector).numpy()
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # Left view: Uncropped full scene tracking your target coordinates
    ax1.imshow(display_img)
    ax1.set_title(f"DINOv3 Full Input View (Query: X={x_patch}, Y={y_patch})")
    ax1.axis('off')
    
    # Draw selected patch marker based on 16px blocks
    patch_size = 16
    center_x = (x_patch * patch_size) + (patch_size // 2)
    center_y = (y_patch * patch_size) + (patch_size // 2)
    ax1.plot(center_x, center_y, marker='o', color='red', markersize=9, markeredgecolor='white', markeredgewidth=2)
    
    # Right view: Heatmap with dynamic normalization bounds
    # Removing hardcoded vmin/vmax allows full-spectrum contrast scaling
    im = ax2.imshow(similarity_map, cmap='magma', interpolation='nearest')
    ax2.set_title("DINOv3 Patch Cosine Similarity Heatmap")
    ax2.axis('off')
    
    cbar = fig.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
    cbar.set_label('Local Similarity Range', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()

In [24]:
# Load PIL Image
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"
pil_image = load_image(url)


# pil_image = Image.open(

# 2. Interactive execution routine
# pil_image = pil_from_array(mock_sensor_data)
arr = np.array(pil_image).transpose(2, 0, 1)
# print(f"{arr.shape=}")
h_trimmed = 16 * int(arr.shape[1] / 16)
w_trimmed = 16 * int(arr.shape[2] / 16)
# arr = arr[:, :h_trimmed, :w_trimmed]
# C, H, W = arr.shape
# print(f"{arr.shape=}")

# 
# w_trimemed = int(W / 16) * 16
# C, H, W = np.asarray(pil_image).shape

# Create image processor
processor = AutoImageProcessor.from_pretrained(model_name)
processor.do_center_crop = True
# processor.do_resize = False
# processor.size = {"height": arr.shape[1], "width": arr.shape[2]} 
processor.crop_size = {"height": h_trimmed, "width": w_trimmed} 
processor.size = {"height": h_trimmed, "width": w_trimmed} 

In [25]:
patch_grid, display_img, grid_h, grid_w = extract_dinov3_features(pil_image, model, processor, device)


# 3. Fire up the interactive environment
interact(
    plot_cosine_similarity,
    x_patch=widgets.IntSlider(min=0, max=grid_w-1, step=1, value=grid_w//2, description='X (Col)'),
    y_patch=widgets.IntSlider(min=0, max=grid_h-1, step=1, value=grid_h//2, description='Y (Row)')
);

interactive(children=(IntSlider(value=30, description='X (Col)', max=59), IntSlider(value=21, description='Y (…

In [26]:
patch_grid, display_img, grid_h, grid_w = extract_dinov3_features(pil_image, model, processor, device)


# 3. Fire up the interactive environment
interact(
    plot_cosine_similarity,
    x_patch=widgets.IntSlider(min=0, max=grid_w-1, step=1, value=grid_w//2, description='X (Col)'),
    y_patch=widgets.IntSlider(min=0, max=grid_h-1, step=1, value=grid_h//2, description='Y (Row)')
);

interactive(children=(IntSlider(value=30, description='X (Col)', max=59), IntSlider(value=21, description='Y (…